# Week 29, Day 4: Properties

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w029d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    @property
    def area(self):
        return self.width * self.height

r = Rectangle(2, 3)
print(r.area)
r.width = 10
print(r.area)
try:
    r.area = 99
except AttributeError as e:
    print("AttributeError:", e)

In [ ]:
class Person:
    def __init__(self, first, last, born):
        self.first = first
        self.last = last
        self.born = born

    @property
    def full_name(self):
        return f"{self.first} {self.last}"

    @property
    def initials(self):
        return f"{self.first[0]}.{self.last[0]}."

    def age_on(self, year):
        return year - self.born

ada = Person("Ada", "Lovelace", 1815)
print(ada.full_name, ada.initials, ada.age_on(1843))
ada.last = "King"
print(ada.full_name)

In [ ]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width          # goes through the setter, so it's checked
        self.height = height

    @property
    def width(self):
        return self._width

    @width.setter
    def width(self, value):
        if value <= 0:
            raise ValueError(f"width must be more than 0, not {value}")
        self._width = value

    @property
    def height(self):
        return self._height

    @height.setter
    def height(self, value):
        if value <= 0:
            raise ValueError(f"height must be more than 0, not {value}")
        self._height = value

    @property
    def area(self):
        return self.width * self.height

r = Rectangle(2, 3)
r.width = 5
print(r.width, r.area)
for bad in [lambda: Rectangle(-1, 3), lambda: setattr(r, "height", 0)]:
    try:
        bad()
    except ValueError as e:
        print("ValueError:", e)
print(r.width, r.height)

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("below absolute zero")
        self._celsius = round(value, 2)

    @property
    def fahrenheit(self):
        return round(self._celsius * 9 / 5 + 32, 2)

    @fahrenheit.setter
    def fahrenheit(self, value):
        self.celsius = (value - 32) * 5 / 9

t = Temperature(20)
t.fahrenheit = 212
print(t.celsius, t.fahrenheit)

In [ ]:
class Account:
    def __init__(self, owner):
        self._owner = owner
        self._balance = 0

    @property
    def owner(self):
        return self._owner

    @property
    def balance(self):
        return self._balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("deposit must be positive")
        self._balance += amount

acc = Account("Ada")
acc.deposit(50)
print(acc.owner, acc.balance)
try:
    acc.balance = 1_000_000
except AttributeError as e:
    print("AttributeError:", e)

> **Key idea.** `@property` makes `obj.name` call a method: use it for values worked out from others, so they never go stale. `@name.setter` runs on `obj.name = value`: check and tidy there, and store in `self._name`. Route `__init__` through the setters. No setter means read-only. Prefer plain attributes until you need control; use methods for actions and anything with arguments or side effects.

### Exercise: A computed circle

Write `Circle(radius)` with properties `diameter`, `area` (π r², rounded to 2 places) and `circumference` (2 π r, rounded to 2 places), all computed from `radius`, which stays a plain attribute.

In [ ]:
import math

class Circle:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"values\"\n    C = ns[\"Circle\"]\n    c = C(2)\n    assert (c.diameter, c.area, c.circumference) == (4, 12.57, 12.57)\n    c.radius = 10\n    assert (c.diameter, c.area) == (20, 314.16)\n    import inspect\n    assert isinstance(inspect.getattr_static(C, \"area\"), property)")

### Exercise: A checked price

Write `Product(name, price)` where `name` and `price` are properties with setters: the name is stripped and must not be empty; the price is a whole number of pence, 0 or more (`True` and floats aren't allowed). Both raise `ValueError` for bad values, including in `__init__`. Add a read-only property `label` like `"Tea £2.50"`.

In [ ]:
class Product:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"values\"\n    P = ns[\"Product\"]\n    p = P(\"  Tea \", 250)\n    assert (p.name, p.price, p.label) == (\"Tea\", 250, \"Tea \u00a32.50\")\n    p.price = 5\n    p.name = \"Mint tea\"\n    assert p.label == \"Mint tea \u00a30.05\"\ndef test_checks():\n    \"checks\"\n    P = ns[\"Product\"]\n    p = P(\"Tea\", 250)\n    for f in [lambda: P(\"\", 1), lambda: P(\"x\", -1), lambda: P(\"x\", 2.5), lambda: setattr(p, \"price\", True), lambda: setattr(p, \"name\", \"   \")]:\n        try:\n            f()\n            assert False, \"should raise ValueError\"\n        except ValueError:\n            pass\n    assert (p.name, p.price) == (\"Tea\", 250)\n    try:\n        p.label = \"x\"\n        assert False, \"label should be read-only\"\n    except AttributeError:\n        pass")

### Exercise: Two ways to set it

Write `Clock(hours, minutes)` storing only `_total` (minutes after midnight, always 0 to 1439, wrapping round). Properties `hours` and `minutes` read and write their part (setting `minutes = 75` adds an hour; setting `hours = 25` wraps to 1), and `total` reads and writes the whole thing (wrapping too). `__repr__` gives `Clock(9, 5)`; `__str__` gives `09:05`.

In [ ]:
class Clock:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"values\"\n    C = ns[\"Clock\"]\n    c = C(9, 5)\n    assert (c.hours, c.minutes, c.total, str(c), repr(c)) == (9, 5, 545, \"09:05\", \"Clock(9, 5)\")\n    c.minutes = 75\n    assert str(c) == \"10:15\"\n    c.hours = 25\n    assert str(c) == \"01:15\"\n    c.total = -30\n    assert str(c) == \"23:30\" and C(24, 0).total == 0 and C(0, -1).total == 1439\n    assert vars(c) == {\"_total\": 1410}, \"store only _total\"")

### Exercise: Look, don't touch

Change `Counter` so `count` can be read but not set from outside (a read-only property over `_count`), while `click()` and `reset()` still work.

In [ ]:
class Counter:
    def __init__(self):
        self.count = 0

    def click(self):
        self.count += 1

    def reset(self):
        self.count = 0

c = Counter()
c.click(); c.click()
print(c.count)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"read-only\"\n    expect_output(\"2\")\n    c = ns[\"Counter\"]()\n    c.click()\n    try:\n        c.count = 100\n        assert False, \"count should be read-only\"\n    except AttributeError:\n        pass\n    c.reset()\n    assert c.count == 0")

### Bug hunt: Forever

Making a `Box` crashes with `RecursionError`. Fix the setter.

In [ ]:
class Box:
    def __init__(self, size):
        self.size = size

    @property
    def size(self):
        return self._size

    @size.setter
    def size(self, value):
        if value < 0:
            raise ValueError("size can't be negative")
        self.size = value

print(Box(3).size)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"works\"\n    expect_output(\"3\")")

### Bug hunt: A method in disguise

This prints `<bound method Order.total of ...>` instead of `£7.50`. Make `total` a property so the last line works as written.

In [ ]:
class Order:
    def __init__(self):
        self.prices = []

    def add(self, pence):
        self.prices.append(pence)

    def total(self):
        p = sum(self.prices)
        return f"£{p // 100}.{p % 100:02d}"

o = Order()
o.add(500); o.add(250)
print(o.total)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a property\"\n    expect_output(\"\u00a37.50\")")

### Bug hunt: The check that's skipped

The setter refuses negative ages, but `Person("Ada", -5)` is accepted, because `__init__` goes round the setter. Fix `__init__`.

In [ ]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self._age = age

    @property
    def age(self):
        return self._age

    @age.setter
    def age(self, value):
        if not 0 <= value <= 150:
            raise ValueError(f"impossible age: {value}")
        self._age = value

try:
    Person("Ada", -5)
    print("accepted")
except ValueError as e:
    print("ValueError:", e)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"refused\"\n    expect_output(\"ValueError: impossible age: -5\")")

### Puzzle: A checked property

Arrange the lines into a `Volume` class whose `level` must be 0 to 10. One line is a trap.

(Put these lines in order on the website.)

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a thermostat schedule

Write `Schedule()` for a heating schedule of up to 6 periods per day. Each period is `(start, end, target)`, times as `"HH:MM"`. Provide:

- `add(start, end, target)`: target 5 to 30; start before end; periods mustn't overlap one another (touching end to start is fine); at most 6 (`ValueError` for any breach, changing nothing);
- a read-only property `periods`: a list of the periods sorted by start time (a **copy**, so changing it can't change the schedule);
- a property `default` (the temperature outside any period, starting at 16, settable within 5 to 30);
- `target_at(time)`: the target at `"HH:MM"` (a period covers its start up to but not including its end);
- a read-only property `hours_heated`: the total length of all periods in hours, as a float rounded to 2 places.

In [ ]:
class Schedule:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_basic():\n    \"periods and targets\"\n    S = ns[\"Schedule\"]\n    s = S()\n    s.add(\"17:00\", \"22:30\", 21)\n    s.add(\"06:30\", \"08:00\", 20)\n    assert s.periods == [(\"06:30\", \"08:00\", 20), (\"17:00\", \"22:30\", 21)]\n    assert s.target_at(\"07:59\") == 20 and s.target_at(\"08:00\") == 16 and s.target_at(\"17:00\") == 21 and s.target_at(\"23:00\") == 16\n    s.default = 18\n    assert s.target_at(\"12:00\") == 18 and s.hours_heated == 7.0\n    copy = s.periods\n    copy.append((\"x\", \"y\", 0))\n    assert len(s.periods) == 2\ndef test_refused():\n    \"refused changes\"\n    S = ns[\"Schedule\"]\n    s = S()\n    s.add(\"06:30\", \"08:00\", 20)\n    s.add(\"08:00\", \"09:00\", 19)\n    for args in [(\"07:00\", \"07:30\", 20), (\"05:00\", \"06:31\", 20), (\"10:00\", \"09:00\", 20), (\"10:00\", \"11:00\", 31), (\"10:00\", \"11:00\", 4)]:\n        try:\n            s.add(*args)\n            assert False, f\"add{args} should raise ValueError\"\n        except ValueError:\n            pass\n    assert len(s.periods) == 2\n    for t in [\"10:00\", \"11:00\", \"12:00\", \"13:00\"]:\n        s.add(t, t[:3] + \"30\", 18)\n    try:\n        s.add(\"20:00\", \"21:00\", 18)\n        assert False, \"a seventh period should raise ValueError\"\n    except ValueError:\n        pass\n    for f in [lambda: setattr(s, \"default\", 31), lambda: setattr(s, \"periods\", []), lambda: setattr(s, \"hours_heated\", 1)]:\n        try:\n            f()\n            assert False, \"should be refused\"\n        except (ValueError, AttributeError):\n            pass")

## Warm-up

`@property` turns the method `area` into something read like an attribute: `r.area`, no brackets. It's worked out fresh on every read, so it can never go stale, and since there's no way to set it, `r.area = 99` is refused.

## Computed attributes

`@property` is a **decorator**: a line starting with `@` above a `def`, which transforms the function defined below it (you've used `@classmethod` and `@cache` already; Phase 4 shows how decorators work inside). `@property` makes reading `obj.name` call the method and give its result.

Use it for values **worked out from other attributes**, that should feel like data rather than an action:

`age_on` stays a method: it needs an argument. A property takes none except `self`.

## Setters that check

A property can also control **assignment**. Write a second method with the same name, decorated with `@name.setter`; it runs whenever someone writes `obj.name = value`. The real value is kept in an **internal attribute**, by convention the same name with an underscore: the property `width` stores into `self._width`.

Three details matter:

- **`__init__` assigns to `self.width`, not `self._width`**, so the setter's check runs for the starting value too. One check, used everywhere.
- **The setter must store into a different name** (`self._width`). Writing `self.width = value` inside the width setter would call the setter again, forever, until `RecursionError`.
- Code using the class doesn't change at all: `r.width = 5` looks exactly like assigning a plain attribute. That's the big advantage: you can start with a plain attribute and turn it into a property later, when you need a check, without changing any code that uses the class.

A setter can also **tidy** values (strip, round, convert), like `__init__` did on Day 3 of last week, and a property's getter can return a converted view of the stored value:

Only one number is stored; `fahrenheit` is a second way to read and write it, and its setter reuses the `celsius` setter's check.

## Read-only attributes

A property with **no setter** can be read but not assigned: Python raises `AttributeError`. That's how to let the outside world **see** a value while only the class's own methods **change** it, keeping the class's rules (Day 4 of last week) in force:

Python can't stop a determined programmer writing `acc._balance = 1_000_000`, but the underscore says clearly that they're breaking the rules. Tomorrow looks at what "private" means in Python.

## Attribute, property or method?

- A **plain attribute** when any value is fine and nothing depends on it: start here. You can always turn it into a property later.
- A **property** when reading should look like data but needs work (a computed value), or assignment needs checking or tidying, or the value should be read-only.
- A **method** when it's an action, takes arguments, changes things, or is slow. Readers assume reading an attribute is quick and changes nothing; a property that downloads a file or changes state breaks that expectation.

## Practice

## Quiz

## Daily challenge

## Before you go

Find an attribute in one of your classes that should never be set to certain values, and turn it into a property with a checking setter. Check that no code using the class had to change. Tomorrow: encapsulation, and what "private" really means in Python.